<a href="https://colab.research.google.com/github/Harsh-Prajapati54/LLMs---Playbook/blob/main/Document_Loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # Document Loader

 in this note book i am going to implement pipline for injection and preprocessing of `pdf` using `PyMupdf `

***Script that ingests a PDF, extracts text, splits into question-level chunks, and saves as structured JSON — zero libraries except PyMuPDF.***

 ### setup

In [1]:
%%capture
!pip install pymupdf

In [2]:
import pymupdf
print(pymupdf.__doc__)

PyMuPDF 1.27.2.2: Python bindings for the MuPDF 1.27.2 library.
Python 3.12 running on linux (64-bit).



In [3]:
%%capture
!pip install -U langchain-text-splitters
!pip install langchain-community

## scripts for opening and preprocessing the pdf document

In [5]:
# loads the document
doc = pymupdf.open("/content/drive/MyDrive/LLm training datasets /Copy of AI Engineering.pdf")

Extracting text from Pdf

In [7]:
doc = pymupdf.open("/content/drive/MyDrive/LLm training datasets /Copy of AI Engineering.pdf")
# cretating a text output
out = open("extracted_text.txt","wb")
# iterating over pages
for page in doc:
  tp = page.get_textpage_ocr()
  text = page.get_text(textpage = tp).encode("utf8") # get plain text (is in UTF-8)
  out.write(text) # write text of pages
  out.write(f"\n--- Page {page.number + 1} ---\n".encode("utf8"))

out.close()

In [8]:
with open("extracted_text.txt","r") as f:
    full_document_content = f.read()
# The `extracted_doc` variable (file object) is no longer needed after reading its content into `full_document_content`.

## Chunking the Text ( Sementic Chunking )


In [23]:
import pymupdf
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import re # Import re for regex operations

def perform_Chunking(text_content, chunk_size=500, chunk_overlap=100):
  """ Perform semantic chunking on the text content using recursive character splitting
    at logical text boundaries.

    Args:
        text_content (str): The text content to process
        chunk_size (int): The target size of each chunk in characters
        chunk_overlap (int): The number of characters of overlap between chunks

    Returns:
        list: The semantically chunked documents with metadata
    """
  # creating text splitter
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap,
      separators = ["\n\n","\n",". "," ",""],
      length_function = len

  )

  # Split the text into semantic chunks (list of strings)
  semantic_chunks_text = text_splitter.split_text(text_content)
  print(f"Document split into {len(semantic_chunks_text)} semantic chunks")


  # Determine section titles for enhanced metadata (simplified for initial fix)
  section_patterns = [
        r'^#+\s+(.+)$',      # Markdown headers
        r'^.+\n[=\-]{2,}$',  # Underlined headers
        r'^[A-Z\s]+:$'       # ALL CAPS section titles
    ]


   # Convert to Document objects with enhanced metadata
  documents = []
  current_section_default = "General Section"

  for i, chunk_text in enumerate(semantic_chunks_text):
        # Simple section detection: check for patterns in the chunk.
        # This can be made more sophisticated to track sections across chunks.
        section_found_in_chunk = False
        for line in chunk_text.split('\n'):
            for pattern in section_patterns:
                match = re.match(pattern, line, re.MULTILINE)
                if match:
                    current_section_default = match.group(0).strip()
                    section_found_in_chunk = True
                    break
            if section_found_in_chunk:
                break

        doc = Document(
            page_content=chunk_text,
            metadata={
                "chunk_id": i,
                "total_chunks": len(semantic_chunks_text),
                "chunk_size": len(chunk_text),
                "chunk_type": "semantic",
                "section": current_section_default # Assign the last found section or default
            }
        )
        documents.append(doc)

  return documents

In [24]:
chunks = perform_Chunking(full_document_content, chunk_size=512, chunk_overlap=100)

Document split into 2894 semantic chunks


In [28]:
 # Display results
print("\n----- CHUNKING RESULTS -----")
print(f"Total semantic chunks: {len(chunks)}")


----- CHUNKING RESULTS -----
Total semantic chunks: 2894


## Exploring Chunks

In [30]:
 # Print an example chunk
print("\n----- EXAMPLE SEMANTIC CHUNK -----")
chunk_idx = 1
example_chunk = chunks[chunk_idx]
print(f"Chunk {chunk_idx} content ({len(example_chunk.page_content)} characters):")
print("-" * 40)
print(example_chunk.page_content)
print("-" * 40)
print(f"Metadata: {example_chunk.metadata}")


----- EXAMPLE SEMANTIC CHUNK -----
Chunk 1 content (480 characters):
----------------------------------------
--- Page 1 ---
9
7 8 1 0 9 8 1 6 6 3 0 4
5 7 9 9 9
ISBN:   978-1-098-16630-4
US $79.99   CAN $99.99
DATA
Foundation models have enabled many new AI use cases while lowering the barriers to entry for  
building AI products. This has transformed AI from an esoteric discipline into a powerful development 
tool that anyone can use—including those with no prior AI experience.
In this accessible guide, author Chip Huyen discusses AI engineering: the process of building applications
----------------------------------------
Metadata: {'chunk_id': 1, 'total_chunks': 2894, 'chunk_size': 480, 'chunk_type': 'semantic', 'section': 'General Section'}


our dataloader is ready !!

In [31]:
for chunk_idx in chunks[:5]:
    print("----")
    print(chunk_idx.page_content)
    print(chunk_idx.metadata)

----
Chip Huyen
 AI Engineering
Building Applications  
with Foundation Models
O'REILLY
7
a
,
yy
eo ye hie
{'chunk_id': 0, 'total_chunks': 2894, 'chunk_size': 101, 'chunk_type': 'semantic', 'section': 'General Section'}
----
--- Page 1 ---
9
7 8 1 0 9 8 1 6 6 3 0 4
5 7 9 9 9
ISBN:   978-1-098-16630-4
US $79.99   CAN $99.99
DATA
Foundation models have enabled many new AI use cases while lowering the barriers to entry for  
building AI products. This has transformed AI from an esoteric discipline into a powerful development 
tool that anyone can use—including those with no prior AI experience.
In this accessible guide, author Chip Huyen discusses AI engineering: the process of building applications
{'chunk_id': 1, 'total_chunks': 2894, 'chunk_size': 480, 'chunk_type': 'semantic', 'section': 'General Section'}
----
with readily available foundation models. AI application developers will discover how to navigate  
the AI landscape, including models, datasets, evaluation benchmarks, and the